In [1]:
import torch
from occhio.autoencoder import TiedLinearRelu
from occhio.distributions.hierarchical import HierarchicalSparse
from occhio.model_grid import ModelGrid, Axis
from occhio.toy_model import ToyModel
from occhio.visualization import *

device = "mps"

In [2]:
N_FEATURES, N_HIDDEN = 5, 2

axis_embed = Axis(label="p_base", values=torch.logspace(-1, 0, 6))

def create_embedding_model(params):
    return ToyModel(
        distribution=HierarchicalSparse(
            N_FEATURES, p_base=float(params["p_base"]), depth_decay=0.85, max_children=3,
            device=device, generator=torch.Generator(device=device).manual_seed(42),
        ),
        importances=0.9 ** torch.arange(N_FEATURES),
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device,
                          generator=torch.Generator(device=device).manual_seed(7)),
    )

grid_embed = ModelGrid(create_embedding_model, axes=[axis_embed])

Grouping distributions: 100%|██████████| 6/6 [00:00<00:00, 3324.42model/s]


In [3]:
grid_embed.fit(batch_size=2048, n_epochs=20_000)

Training:   3%|▎         | 548/20000 [00:02<01:41, 191.67epoch/s]


KeyboardInterrupt: 

In [ ]:
print(axis_embed.values)
plot_embedding(grid_embed)


In [4]:
N_FEATURES, N_HIDDEN = 5, 2

axis_importance = Axis(label="Importance=x_i", values=torch.logspace(-1, 0, 70))
axis_density    = Axis(label="p_base",     values=torch.logspace(-1, 0,  70))

In [5]:
def create_phase_model(params):
    return ToyModel(
        distribution=HierarchicalSparse(
            N_FEATURES, p_base=float(params["p_base"]), depth_decay=0.85, max_children=3,
            device=device, generator=torch.Generator(device=device).manual_seed(42),
        ),
        importances=float(params["Importance=x_i"]) ** torch.arange(N_FEATURES),
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device,
                          generator=torch.Generator(device=device).manual_seed(7)),
    )

In [6]:
# grid_phase = ModelGrid(create_phase_model, axes=[axis_importance, axis_density], cache_samples=True)
# grid_phase.fit(batch_size=216, n_epochs=10_000)
# grid_phase.save_models("phase_23022026_14_25_DONOTTOUCH.pkl")

In [8]:
grid_phase_cached = ModelGrid(create_phase_model, axes=[axis_importance, axis_density])
grid_phase_cached.load_models("phase_23022026_14_25_DONOTTOUCH.pkl")
grid_phase_cached.axes[0].label = "Relative Importance"

Grouping distributions: 100%|██████████| 4900/4900 [00:00<00:00, 6748.97model/s]


In [9]:
fig= plot_phase_change_multi(grid_phase_cached, up_to=5)
fig.show()

In [10]:
fig.write_image("phase_change_multi.png")

In [ ]:
import numpy as np

def get_weights_equal_map(original_grid, cached_grid, rtol=1e-2, atol=0.5):
    org_models = original_grid.models.ravel()
    cached_models = cached_grid.models.ravel()
    n = min(len(org_models), len(cached_models))
    bool_map = []
    for i in range(n):
        w_org = org_models[i].W.detach().cpu().numpy()
        w_cached = cached_models[i].W.detach().cpu().numpy()
        bool_map.append(np.allclose(w_org, w_cached, rtol=rtol, atol=atol))
    return np.array(bool_map)

weights_equal_map = get_weights_equal_map(grid_phase, grid_phase_cached, rtol=1e-2, atol=0.5)
print(weights_equal_map)

print("First 5 model weights from grid_phase:")
for i in range(5):
    w = grid_phase.models.ravel()[i].W.detach().cpu().numpy()
    print(f"grid_phase model {i} weights:\n{w}\n")

print("First 5 model weights from grid_phase_cached:")
for i in range(5):
    w = grid_phase_cached.models.ravel()[i].W.detach().cpu().numpy()
    print(f"grid_phase_cached model {i} weights:\n{w}\n")


In [ ]:
# Exp 3 — Geometry: 100 features, 20 hidden (5:1), sweep p_base
# Flat importance so geometry is driven by density alone
N_FEATURES, N_HIDDEN = 100, 20

axis_geo = Axis(label="p_base", values=torch.logspace(-2, 0, 16))

def create_geometry_model(params):
    return ToyModel(
        distribution=HierarchicalSparse(
            N_FEATURES, p_base=float(params["p_base"]), depth_decay=0.85, max_children=4,
            device=device, generator=torch.Generator(device=device).manual_seed(42),
        ),
        importances=0.999 ** torch.arange(N_FEATURES),
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device,
                          generator=torch.Generator(device=device).manual_seed(7)),
    )

grid_geo = ModelGrid(create_geometry_model, axes=[axis_geo])

In [ ]:
grid_geo.fit(batch_size=216, n_epochs=10_000)

In [ ]:
plot_geometry(grid_geo)